# Qwen: export a model and run a structured question on CPU

**You are here → 05 — Qwen with ExecuTorch.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| **→ 5. CPU portability (optional) (current)** | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| 6. Compatibility study (optional) | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

The serving chapters used an HTTP API. Here we package the same **Qwen3-4B** base revision as an **ExecuTorch XNNPACK** program and invoke its native CPU runner. The recipe uses **INT4 linear weights with dynamic INT8 activations and INT8 embeddings**, which differs from AWQ W4A16.

We test one direct clinical SQL question with a short schema. This is a portability smoke, separate from the full planning/repair workload and GPU serving benchmarks.

Configure the edge environment below, then run the cells in order. Reuse a validated export explicitly or perform conversion and export. **Generation always runs afresh in Live mode.** Install the [notebook environment](../SETUP.md). See [the edge setup guide](../tutorial/EXECUTORCH_MAC.md).


## 1. Imports and editable configuration

`MODE="live"` performs the stages below. Choose **plan** to inspect without edge execution or **replay** to read the explicitly selected prior record. The notebook mode overrides an old local JSON mode. No stage installs packages or downloads weights implicitly.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.edge_workflow import EdgeStudy

MODE = "live"  # "plan" previews stages; "replay" displays archived results
MODE = os.environ.get('LOCAL_AI_LAB_MODE', MODE)  # Explicit CPU validation override.
# Machine settings: tutorial/config/edge-qwen.local.json
study = EdgeStudy('qwen', root=ROOT, mode=MODE)
print('Effective mode:', study.mode)
print('Model:', 'Qwen/Qwen3-4B')
print('Hardware target: local CPU / separate ExecuTorch environment')
print('Session:', study.session_id)
print('Current-session measurements:', len(study.results))
display(JSON(study.config))


Effective mode: live
Model: Qwen/Qwen3-4B
Hardware target: local CPU / separate ExecuTorch environment
Session: 20261110T121532-73d94917db
Current-session measurements: 0


<IPython.core.display.JSON object>

## 2. Preflight

Check the configured edge Python and installed runtime before any model work. Qwen requires a pinned source checkpoint and either a validated reusable export or explicit conversion/export. Nemotron needs only its pinned configuration. Missing prerequisites stop with instructions.


In [2]:
display(JSON(study.preflight()))


<IPython.core.display.JSON object>

## 3. Prepare and validate the CPU program

Stages run as separate processes so large reference and converted models do not stay resident together. `download_models=false` uses staged source weights. Set it to true only to request a download.

With `reuse_export=true`, conversion, reference, validation and export cells verify the recorded artifact hashes and report **reused_artifact**. They do not claim a new export. To build afresh, set `reuse_export=false` and select a new `artifact_dir` in the local config. Conversion uses the official Qwen3 weight mapping; FP32 logit checks must pass before quantized export.

The native recipe and generation call are in [run_qwen_edge.py](../scripts/tutorial/run_qwen_edge.py).


In [3]:
stage_result = study.stage('download')
display(JSON(stage_result))


Starting edge stage: download | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

In [4]:
stage_result = study.stage('prepare')
display(JSON(stage_result))


Starting edge stage: prepare | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

In [5]:
stage_result = study.stage('reference')
display(JSON(stage_result))


Starting edge stage: reference | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

In [6]:
stage_result = study.stage('validate')
display(JSON(stage_result))


Starting edge stage: validate | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

In [7]:
stage_result = study.stage('export')
display(JSON(stage_result))


Starting edge stage: export | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

## 4. Invoke the model and check its answer

The next cell starts a **new native generation process** with the verified `.pte` program and tokenizer. The runner calls `TextLLMRunner.generate()` with temperature zero, a short clinical schema and an output cap of 192 tokens. It uses no HTTP serving API and loads no saved answer. Model-load time is separate from inference timing.

The following cell executes the newly generated SQL in DuckDB and compares its rows with the reference. SQL scoring occurs after native timing.


In [8]:
edge = study.stage('run')
display(JSON(edge))


Starting edge stage: run | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

In [9]:
sql_check = study.stage('sql_check')
display(JSON(sql_check))


Starting edge stage: sql_check | session 20261110T121532-73d94917db


<IPython.core.display.JSON object>

## 5. Read this session's measurements

Compare model loading, first-token time, inference latency, artifact footprint and process RSS. These are one short-question sample, not serving percentiles. RSS is CPU process memory, not GPU memory. Inspect generated SQL and its reference-result check alongside timing.


In [10]:
current = study.results.get('run', {})
if current.get('session_id') == study.session_id and current.get('status') == 'completed':
    observer = current.get('native_observer') or {}
    ttft = ((observer['first_token_ms'] - observer['inference_start_ms']) / 1000
            if 'first_token_ms' in observer and 'inference_start_ms' in observer else None)
    display(JSON({'session': study.session_id,
        'model_load_seconds': current.get('startup_seconds'), 'ttft_seconds': ttft,
        'inference_seconds': current.get('single_request_seconds'),
        'footprint_bytes': current.get('footprint_bytes'), 'memory': current.get('memory'),
        'generation': current.get('generation'),
        'sql_check': study.results.get('sql_check')}))
else:
    print('Native measurements are unavailable until generation completes in this session.')
display(Markdown(study.report()))


<IPython.core.display.JSON object>

**Mode: live · Session: 20261110T121532-73d94917db**

| Stage | Status | Evidence |
|---|---|---|
| download | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/download-9d9f59a0.json |
| prepare | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/prepare-1d98828c.json |
| reference | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/reference-a29d110a.json |
| validate | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/validate-b6d0a0cb.json |
| export | reused_artifact | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/export-69487e6c.json |
| run | completed | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/run-c0fa578c.json |
| sql_check | completed | experiment-tracker/tutorial/edge/sessions/20261110T121532-73d94917db/sql_check-da715168.json |

## Cleanup

The edge stages execute in foreground child processes and retain no serving process. Save the notebook to retain genuine current outputs. Failures stay in the session evidence.


In [11]:
print(study.close())


Edge stages run in foreground child processes; no serving process is retained.


## Reference: previously measured results

The [edge guide](../tutorial/EXECUTORCH_MAC.md) preserves the previous Qwen native run and Nemotron compatibility check. Choose `MODE="replay"` explicitly to inspect those records. The saved artifacts and historical records remain unchanged; they are not results from a new Live execution.
